# DataConnect solucions - Exploração dos dados

## Objetivo

Realizar o **entendimento inicial dos dados** antes de qualquer transformação ou análise exploratória.

Neste notebook vamos:

1. Identificar os arquivos disponíveis;
2. Carregar os datasets sem alterar os arquivos brutos;
3. Inspecionar dimensões e colunas;
4. Avaliar tipos de dados;
5. Identificar valores ausentes;
6. Verificar duplicidades;
7. Avaliar cardinalidade das chaves;
8. Observar inconsistências preliminares;
9. Documentar hipóteses e pontos de atenção.

> **Princípio analítico:** neste primeiro momento não vamos "corrigir" os dados. O objetivo é diagnosticar a qualidade e a estrutura dos dados para que as decisões de tratamento sejam justificadas no próximo notebook.

### Fontes disponíbilizadas

- `dc_analistas.csv`
- `dc_apontamentos.csv`
- `dc_clientes.csv`
- `dc_projetos.csv`
- `dc_satisfacao.csv`


## 1. Contexto analítico

O conjunto de dados aparenta representar uma operação composta por:

- **Analistas**, responsáveis pela execução das atividades;
- **Apontamentos**, que registram o esforço realizado;
- **Clientes**, que representam a carteira atendida;
- **Projetos**, os registros dos projetos desenvolvidos;
- **Satisfação**, que representam a

Ainda não vamos assumir os relacionamentos completos entre essas entidades.



In [1]:
# @title ## 0. Set Up
!pip install dash plotly pandas sqlite-utils

In [2]:
# @title ## 1. Importação das Bibliotecas
import pandas as pd
import sqlite3
from dash import Dash, dcc, html, Input, Output
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats as st
from IPython.display import Markdown
from IPython.core.display import HTML

import os
import math
import glob
import itertools

from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# @title ## 2. Carregamentos dos Dados
df_clientes = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/mini_projeto/data/raw/dc_clientes.csv')
df_apontamentos = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/mini_projeto/data/raw/dc_apontamentos.csv')
df_projetos = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/mini_projeto/data/raw/dc_projetos.csv')
df_analistas = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/mini_projeto/data/raw/dc_analistas.csv')
df_satisfacao = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/mini_projeto/data/raw/dc_satisfacao.csv')

In [4]:
# @title ## 3. Validação dos Dados
print("=" * 60)
print("VALIDAÇÃO DOS DADOS")
print("=" * 60)
print(f"df_clientes: {df_clientes.shape}")
print(f"df_apontamentos: {df_apontamentos.shape}")
print(f"df_projetos: {df_projetos.shape}")
print(f"df_satisfacao: {df_satisfacao.shape}")
print(f"df_analistas: {df_analistas.shape}")


VALIDAÇÃO DOS DADOS
df_clientes: (62, 7)
df_apontamentos: (9879, 6)
df_projetos: (141, 11)
df_satisfacao: (103, 5)
df_analistas: (24, 6)


In [5]:
# @title ## 4. Visão geral dos datasets



In [6]:
# @title ### 4.1 Informações Iniciais do dataset - clientes
display(Markdown('### **Primeiras Linhas do Dataset**'))
display(df_clientes.head())
print("=" * 80)
display(Markdown('### **Informações do Dataset**'))
display(df_clientes.info())
print("=" * 80)
display(Markdown('### **Quantidade de Linhas e Colunas do Dataset**'))
display(df_clientes.shape)
print("=" * 80)
display(Markdown('### **Quantidade de Valores Ausentes**'))
display(df_clientes.isnull().sum())
print("=" * 80)
display(Markdown('### **Quantidade de Valores Duplicados**'))
display(df_clientes.duplicated().sum())
print("=" * 80)
display(Markdown('### **Quantidade de Valores Únicos**'))
display(df_clientes.nunique())
print("=" * 80)
display(Markdown('### **Estatísticas Descritivas**'))
display(df_clientes.describe())
print("=" * 80)
display(Markdown('### **Os tipos de dados**'))
display(df_clientes.dtypes)

### **Primeiras Linhas do Dataset**

,cliente_id,cliente,setor,porte,cidade,uf,data_cadastro
0,CLI001,Umbu S.A.,NaN,Pequeno,Rio de Janeiro,Rio de Janeiro,2021-10-13
1,CLI002,Xingu S.A.,Indústria,Médio,Florianópolis,SC,2021-03-07
2,CLI003,Aurora S.A.,Saúde,Médio,Curitiba,PR,2021-02-24
3,CLI004,Rubi Ltda,Logística,Pequeno,Florianópolis,SC,2024-04-21
4,CLI005,Ipê S.A.,Logística,Pequeno,Belo Horizonte,MG,15/11/2021


### **Informações do Dataset**

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62 entries, 0 to 61
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   cliente_id     62 non-null     object
 1   cliente        62 non-null     object
 2   setor          57 non-null     object
 3   porte          62 non-null     object
 4   cidade         62 non-null     object
 5   uf             62 non-null     object
 6   data_cadastro  62 non-null     object
dtypes: object(7)
memory usage: 3.5+ KB


None

...

,0
cliente_id,60
cliente,60
setor,17
porte,3
cidade,10
uf,19
data_cadastro,60


### **Estatísticas Descritivas**

,cliente_id,cliente,setor,porte,cidade,uf,data_cadastro
count,62,62,57,62,62,62,62
unique,60,60,17,3,10,19,60
top,CLI004,Rubi Ltda,Saúde,Pequeno,São Paulo,SP,2024-04-21
freq,2,2,7,38,11,11,2


### **Os tipos de dados**

,0
cliente_id,object
cliente,object
setor,object
porte,object
cidade,object
uf,object
data_cadastro,object


In [7]:
# @title ### 4.2 Informações Iniciais do dataset - Análistas
display(Markdown('### **Primeiras Linhas do Dataset**'))
display(df_analistas.head())
print("=" * 80)
display(Markdown('### **Informações do Dataset**'))
display(df_analistas.info())
print("=" * 80)
display(Markdown('### **Quantidade de Linhas e Colunas do Dataset**'))
display(df_analistas.shape)
print("=" * 80)
display(Markdown('### **Quantidade de Valores Ausentes**'))
display(df_analistas.isnull().sum())
print("=" * 80)
display(Markdown('### **Quantidade de Valores Duplicados**'))
display(df_analistas.duplicated().sum())
print("=" * 80)
display(Markdown('### **Quantidade de Valores Únicos**'))
display(df_analistas.nunique())
print("=" * 80)
display(Markdown('### **Estatísticas Descritivas**'))
display(df_analistas.describe())
print("=" * 80)
display(Markdown('### **Os tipos de dados**'))
display(df_analistas.dtypes)

### **Primeiras Linhas do Dataset**

,analista_id,analista,squad,senioridade,custo_hora,data_admissao
0,ANL001,Ana Barbosa,ALPHA,Senior,"150,00",2021-12-07
1,ANL002,Bruno Ipiranga,Bravo,Pleno,95.00,2021-08-30
2,ANL003,Carla Esteves,Charlie,Junior,55.00,2024-12-26
3,ANL004,Diego Freitas,Delta,Especialista,210.00,2025-07-28
4,ANL005,Elisa Werneck,Echo,Pleno,95.00,24/10/2023


### **Informações do Dataset**

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   analista_id    24 non-null     object
 1   analista       24 non-null     object
 2   squad          24 non-null     object
 3   senioridade    24 non-null     object
 4   custo_hora     24 non-null     object
 5   data_admissao  24 non-null     object
dtypes: object(6)
memory usage: 1.3+ KB


None

...

,0
analista_id,24
analista,24
squad,9
senioridade,4
custo_hora,7
data_admissao,24


### **Estatísticas Descritivas**

,analista_id,analista,squad,senioridade,custo_hora,data_admissao
count,24,24,24,24,24,24
unique,24,24,9,4,7,24
top,ANL001,Ana Barbosa,Bravo,Junior,55.00,2021-12-07
freq,1,1,4,7,7,1


### **Os tipos de dados**

,0
analista_id,object
analista,object
squad,object
senioridade,object
custo_hora,object
data_admissao,object


In [8]:
# @title ### 4.3 Informações Iniciais do dataset - Projetos
display(Markdown('### **Primeiras Linhas do Dataset**'))
display(df_projetos.head())
print("=" * 80)
display(Markdown('### **Informações do Dataset**'))
display(df_projetos.info())
print("=" * 80)
display(Markdown('### **Quantidade de Linhas e Colunas do Dataset**'))
display(df_projetos.shape)
print("=" * 80)
display(Markdown('### **Quantidade de Valores Ausentes**'))
display(df_projetos.isnull().sum())
print("=" * 80)
display(Markdown('### **Quantidade de Valores Duplicados**'))
display(df_projetos.duplicated().sum())
print("=" * 80)
display(Markdown('### **Quantidade de Valores Únicos**'))
display(df_projetos.nunique())
print("=" * 80)
display(Markdown('### **Estatísticas Descritivas**'))
display(df_projetos.describe())
print("=" * 80)
display(Markdown('### **Os tipos de dados**'))
display(df_projetos.dtypes)

### **Primeiras Linhas do Dataset**

,projeto_id,cliente_id,nome_projeto,tipo_servico,squad,status,data_inicio,data_fim_prevista,data_fim_real,horas_vendidas,valor_contrato
0,PRJ0001,CLI013,Dashboard BI - Windsor,Dashboard B.I.,Delta,CONCLUÍDO,2024-03-19,2024-05-06,08/05/2024,288,NaN
1,PRJ0002,CLI022,Diagnóstico de Dados - Umbu,Diagnóstico de Dados,Alpha,Cancelado,2024-10-01,2024-10-28,NaN,89,24044.51
2,PRJ0003,CLI051,Pipeline de Dados - Cristal,Pipeline de Dados,Charlie,Concluído,2025-06-16,20/10/2025,10/15/2025,423,91525.58
3,PRJ0004,CLI060,Dashboard BI - Kairós,Dashboard BI,Echo,Concluído,2024-03-19,25-abr-2024,2024-04-26,220,47409.01
4,PRJ0005,CLI020,Modelo Preditivo - Duna,Modelo Preditivo,Echo,Concluído,09/12/2024,02/27/2025,2025-03-12,338,"61.970,49"


### **Informações do Dataset**

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 141 entries, 0 to 140
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   projeto_id         141 non-null    object
 1   cliente_id         141 non-null    object
 2   nome_projeto       141 non-null    object
 3   tipo_servico       141 non-null    object
 4   squad              141 non-null    object
 5   status             141 non-null    object
 6   data_inicio        141 non-null    object
 7   data_fim_prevista  141 non-null    object
 8   data_fim_real      128 non-null    object
 9   horas_vendidas     141 non-null    int64 
 10  valor_contrato     134 non-null    object
dtypes: int64(1), object(10)
memory usage: 12.2+ KB


None

...

,0
projeto_id,140
cliente_id,54
nome_projeto,87
tipo_servico,20
squad,6
status,5
data_inicio,132
data_fim_prevista,131
data_fim_real,117
horas_vendidas,124


### **Estatísticas Descritivas**

,horas_vendidas
count,141.000000
mean,327.382979
std,225.486447
min,51.000000
25%,149.000000
50%,285.000000
75%,430.000000
max,1030.000000


### **Os tipos de dados**

,0
projeto_id,object
cliente_id,object
nome_projeto,object
tipo_servico,object
squad,object
status,object
data_inicio,object
data_fim_prevista,object
data_fim_real,object
horas_vendidas,int64


In [9]:
# @title ### 4.4 Informações Iniciais do dataset - Apontamentos
display(Markdown('### **Primeiras Linhas do Dataset**'))
display(df_apontamentos.head())
print("=" * 80)
display(Markdown('### **Informações do Dataset**'))
display(df_apontamentos.info())
print("=" * 80)
display(Markdown('### **Quantidade de Linhas e Colunas do Dataset**'))
display(df_apontamentos.shape)
print("=" * 80)
display(Markdown('### **Quantidade de Valores Ausentes**'))
display(df_apontamentos.isnull().sum())
print("=" * 80)
display(Markdown('### **Quantidade de Valores Duplicados**'))
display(df_apontamentos.duplicated().sum())
print("=" * 80)
display(Markdown('### **Quantidade de Valores Únicos**'))
display(df_apontamentos.nunique())
print("=" * 80)
display(Markdown('### **Estatísticas Descritivas**'))
display(df_apontamentos.describe())
print("=" * 80)
display(Markdown('### **Os tipos de dados**'))
display(df_apontamentos.dtypes)

### **Primeiras Linhas do Dataset**

,apontamento_id,projeto_id,analista_id,data,horas,atividade
0,APT002473,PRJ0032,ANL006,2026-03-23,6,Reunião com cliente
1,APT004798,PRJ0064,ANL008,2026-05-08,6.5,Apresentação
2,APT005566,PRJ0079,ANL012,2025-06-02,6,Coleta
3,APT000099,PRJ0003,ANL009,2025-09-15,4.5,Documentação
4,APT009367,PRJ0134,ANL009,2026-01-22,4,Modelagem


### **Informações do Dataset**

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9879 entries, 0 to 9878
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   apontamento_id  9879 non-null   object
 1   projeto_id      9879 non-null   object
 2   analista_id     9879 non-null   object
 3   data            9879 non-null   object
 4   horas           9879 non-null   object
 5   atividade       9879 non-null   object
dtypes: object(6)
memory usage: 463.2+ KB


None

...

,0
apontamento_id,9799
projeto_id,140
analista_id,24
data,2037
horas,23
atividade,15


### **Estatísticas Descritivas**

,apontamento_id,projeto_id,analista_id,data,horas,atividade
count,9879,9879,9879,9879,9879,9879
unique,9799,140,24,2037,23,15
top,APT002135,PRJ0042,ANL005,2026-06-08,4,Coleta
freq,2,273,538,57,1667,1232


### **Os tipos de dados**

,0
apontamento_id,object
projeto_id,object
analista_id,object
data,object
horas,object
atividade,object


In [10]:
# @title ### 4.5 Informações Iniciais do dataset - Satisfação
display(Markdown('### **Primeiras Linhas do Dataset**'))
display(df_satisfacao.head())
print("=" * 80)
display(Markdown('### **Informações do Dataset**'))
display(df_satisfacao.info())
print("=" * 80)
display(Markdown('### **Quantidade de Linhas e Colunas do Dataset**'))
display(df_satisfacao.shape)
print("=" * 80)
display(Markdown('### **Quantidade de Valores Ausentes**'))
display(df_satisfacao.isnull().sum())
print("=" * 80)
display(Markdown('### **Quantidade de Valores Duplicados**'))
display(df_satisfacao.duplicated().sum())
print("=" * 80)
display(Markdown('### **Quantidade de Valores Únicos**'))
display(df_satisfacao.nunique())
print("=" * 80)
display(Markdown('### **Estatísticas Descritivas**'))
display(df_satisfacao.describe())
print("=" * 80)
display(Markdown('### **Os tipos de dados**'))
display(df_satisfacao.dtypes)

### **Primeiras Linhas do Dataset**

,pesquisa_id,projeto_id,data_pesquisa,nota_nps,comentario
0,PSQ0001,PRJ0001,2024-05-27,NaN,"Resultado ok, prazo apertado."
1,PSQ0002,PRJ0003,2025-10-20,10.0,Documentação impecável.
2,PSQ0003,PRJ0004,2024-05-08,9.0,Documentação impecável.
3,PSQ0004,PRJ0005,2025-03-27,7.0,"Bom projeto, comunicação pode melhorar."
4,PSQ0005,PRJ0006,17/07/2025,6.0,Suporte demorou a responder.


### **Informações do Dataset**

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   pesquisa_id    103 non-null    object 
 1   projeto_id     103 non-null    object 
 2   data_pesquisa  103 non-null    object 
 3   nota_nps       95 non-null     float64
 4   comentario     103 non-null    object 
dtypes: float64(1), object(4)
memory usage: 4.2+ KB


None

...

,0
pesquisa_id,103
projeto_id,103
data_pesquisa,100
nota_nps,8
comentario,10


### **Estatísticas Descritivas**

,nota_nps
count,95.000000
mean,8.421053
std,1.601721
min,3.000000
25%,8.000000
50%,9.000000
75%,10.000000
max,10.000000


### **Os tipos de dados**

,0
pesquisa_id,object
projeto_id,object
data_pesquisa,object
nota_nps,float64
comentario,object


In [11]:
# @title ## 5. Primeiras observações

A inspeção visual  podemos identificar:

- espaços extras;
- grafias diferentes;
- valores aparentemente numéricos armazenados como texto;
- formatos diferentes de data;
- valores ausentes;
- registros aparentimente duplicados.


In [12]:
df_satisfacao.head(3)

,pesquisa_id,projeto_id,data_pesquisa,nota_nps,comentario
0,PSQ0001,PRJ0001,2024-05-27,NaN,"Resultado ok, prazo apertado."
1,PSQ0002,PRJ0003,2025-10-20,10.0,Documentação impecável.
2,PSQ0003,PRJ0004,2024-05-08,9.0,Documentação impecável.


In [13]:
# @title ## 6. Avaliação preliminar das chaves primarias e estrangeiras

display(Markdown('### **Avaliação das chaves primarias e estrangeiras**'))
chaves_primaria = {
    'df_clientes': 'cliente_id',
    'df_apontamentos': 'apontamento_id',
    'df_projetos': 'projeto_id',
    'df_satisfacao': 'pesquisa_id',
    'df_analistas': 'analista_id',
}
chaves_estrangeiras = {
    'df_apontamentos': ['projeto_id','analista_id'],
    'df_projetos': 'cliente_id',
    'df_satisfacao': 'projeto_id',

}
display(Markdown('### Chaves primarias'))
display(chaves_primaria)

display(Markdown('### Chaves estrangeiras'))
display(chaves_estrangeiras)

### **Avaliação das chaves primarias e estrangeiras**

### Chaves primarias

{'df_clientes': 'cliente_id',
 'df_apontamentos': 'apontamento_id',
 'df_projetos': 'projeto_id',
 'df_satisfacao': 'pesquisa_id',
 'df_analistas': 'analista_id'}

### Chaves estrangeiras

{'df_apontamentos': ['projeto_id', 'analista_id'],
 'df_projetos': 'cliente_id',
 'df_satisfacao': 'projeto_id'}

In [14]:
# @title ## 7. Cardinalidade das potenciais relações




### Analistas → Apontamentos

Um analista pode registrar vários apontamentos.

Esperamos uma relação:

`1 analista → N apontamentos`

### Projetos → Apontamentos

A relação também parece potencialmente ser:

`1 projeto → N apontamentos`


### Clientes → Projetos

Um Cliente pode ter vários projetos.

`1 cliente → N projetos`

### Satisfaçao → Projetos

A pesquisa de satisfação para cada projetos.

`1 pesquisa → 1 projetos`


In [15]:
# @title ## 8. Perguntas de negócio para a próxima etapa



### Capacidade e operação
1. Como as horas trabalhadas estão distribuídas entre analistas?
2. Como o esforço está distribuído entre squads?
3. Quais atividades concentram maior volume de horas?

### Custos
4. Qual é o custo estimado do esforço registrado?
5. Quais squads concentram maior custo?
6. Existe concentração de custo em determinados analistas ou projetos?

### Projetos e clientes
7. Quais projetos demandam maior esforço?
8. Como o esforço se distribui entre clientes?
9. Existe relação entre esforço, custo e satisfação?